# CS336 Spring 2026 - File-based `train_bpe()` + 多进程 Pre-tokenization
----
本节目标只有一个：
```text
input_path → binary file → safe chunk boundaries
→ local pre-token Counter → Counter aggregation
→ PretokenCounts → BPE core → vocab + ordered merges
```

核心不是“怎么开进程”，而是先保证：

$$
C_{\text{whole}}=\sum_i C_{\text{safe chunk }i}
$$

任意切块并不满足这个等式。

In [ ]:
from __future__ import annotations

import io
import os
import random
import tempfile
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import regex

SPECIAL = "<|endoftext|>"
DELIMITER = SPECIAL.encode("utf-8")
TokenSequence = tuple[bytes, ...]
PretokenCounts = Counter[TokenSequence]

GPT2_PRETOKEN_PATTERN = (
    r"'(?:[sdmt]|ll|ve|re)"
    r"| ?\p{L}+"
    r"| ?\p{N}+"
    r"| ?[^\s\p{L}\p{N}]+"
    r"|\s+(?!\S)"
    r"|\s+"
)


def pretokenize(text: str) -> list[str]:
    return [m.group(0) for m in regex.finditer(GPT2_PRETOKEN_PATTERN, text)]


def to_byte_tokens(piece: str) -> TokenSequence:
    return tuple(bytes([x]) for x in piece.encode("utf-8"))


def count_ordinary(text: str) -> PretokenCounts:
    return Counter(to_byte_tokens(piece) for piece in pretokenize(text))


def count_with_special_isolation(text: str) -> PretokenCounts:
    # 与 04 一样：先隔离 special-token，普通片段才进入 regex。
    counts = Counter()
    for ordinary_span in text.split(SPECIAL):
        counts.update(count_ordinary(ordinary_span))
    return counts


assert "".join(pretokenize(" Hello 世界 🌍!")) == " Hello 世界 🌍!"

print("基础预分词与 byte representation：通过")

基础预分词与 byte representation：通过


## 1. 本节的困难
> 本节真正的困难是: 文件应该在哪切?

假设完整文本为 $T$, 切成
$$
T=T_1+T_2+\cdots+T_k
$$
希望有
$$
C(T)=\sum_{i=1}^k C(T_i)
$$
其中 $C$ 表示 pre-tokenization+frequency counting.
> 但是任意切分文件则会破坏这个条件, 因此后续要解决此问题.


例如, 完整的 `" wonderful"` 是一个 pre-token, 但要是切成 `" won"`和 `"derful"` 就变成了两个. 因此必须分布验证:
- UTF-8 正确性: 每块能解码, 并且重组时字节不丢失不重复
- pre-tokenization 语义正确性: 分块后 Counter 与完整处理相同.

In [ ]:
whole = " wonderful"
arbitrary_parts = [" won", "derful"]
print("完整:", pretokenize(whole))
print("任意分块:", [part for text in arbitrary_parts for part in pretokenize(text)])
assert pretokenize(whole) != [
    part for text in arbitrary_parts for part in pretokenize(text)
]
print("UTF-8字节:", list("你".encode()))
try:
    "你".encode()[:1].decode("utf-8")
except UnicodeDecodeError:
    print("UTF-8 半字符边界：正确地触发了 UnicodeDecodeError")


完整: [' wonderful']
任意分块: [' won', 'derful']
UTF-8字节: [228, 189, 160]
UTF-8 半字符边界：正确地触发了 UnicodeDecodeError


## 2. 安全边界
本节利用 `<|endoftext|>` 作为 document delimiter(文档分隔符). 因为 special token 在普通 BPE 统计前就被隔离, 不应该跨越该边界进行 BPE merge.

官方实例的思路大概是:
```text
文件大小 → 等距估计 byte offsets
→ 从每个内部 offset 向后找 delimiter
→ 移到 delimiter 起点
→ sorted(set(boundaries))
```

多个估计位置可能落到同一个 delimiter, 所以实际 chunk 数可能少于请求数.

In [3]:
def find_chunk_boundaries(
    file,
    desired_num_chunks: int,
    split_special_token: bytes,
    *,
    read_ahead: int = 4096,
) -> list[int]:
    """在不切断 document delimiter 的位置建立 byte boundaries。"""
    if desired_num_chunks < 1:
        raise ValueError("desired_num_chunks must be >= 1")
    if not isinstance(split_special_token, bytes) or not split_special_token:
        raise ValueError("split_special_token must be nonempty bytes")
    if read_ahead < 1:
        raise ValueError("read_ahead must be >= 1")

    file.seek(0, os.SEEK_END)
    file_size = file.tell()
    file.seek(0)

    approx_size = file_size // desired_num_chunks
    boundaries = [i * approx_size for i in range(desired_num_chunks + 1)]
    boundaries[-1] = file_size

    keep = len(split_special_token) - 1

    for i in range(1, len(boundaries) - 1):
        position = boundaries[i]
        file.seek(position)
        tail = b""

        while True:
            block = file.read(read_ahead)
            if not block:
                boundaries[i] = file_size
                break

            searchable = tail + block
            found = searchable.find(split_special_token)
            if found >= 0:
                boundaries[i] = position - len(tail) + found
                break

            tail = searchable[-keep:] if keep else b""
            position += len(block)

    return sorted(set(boundaries))


## 3. 故意将 delimiter 放在读取块边界
测试时将 `read_ahead` 缩小到5bytes, 但是 `<|endoftext|>` 本身有13bytes. 这样它必然可能横跨多个读取块, 可以用来测试程序.
- `b"".join(chunks) == original`: 不能丢失或重复字节
- 每个 chunk 能完整的 UTF-8 decode
- 内部 boundary 对应 delimiter 的起点

In [4]:
raw = ("开头" + SPECIAL + "alpha 🌍 " + SPECIAL + "尾部").encode("utf-8")
with io.BytesIO(raw) as file:
    boundaries = find_chunk_boundaries(file, 7, DELIMITER, read_ahead=5)

chunks = [raw[a:b] for a, b in zip(boundaries[:-1], boundaries[1:])]

assert b"".join(chunks) == raw
assert all(chunk.decode("utf-8").encode("utf-8") == chunk for chunk in chunks)
assert all(raw[b:].startswith(DELIMITER) for b in boundaries[1:-1])

print("boundaries:", boundaries)
print("chunks:", [x.decode("utf-8") for x in chunks])
print("跨读取缓冲区、UTF-8、重建：通过")

boundaries: [0, 30, 49]
chunks: ['开头<|endoftext|>alpha 🌍 ', '<|endoftext|>尾部']
跨读取缓冲区、UTF-8、重建：通过


## 4. 测试矩阵：不要只测 happy path

检查以下情况：空文件、没有分隔符、连续分隔符、Unicode / emoji、delimiter 横跨读取缓冲区、期望 chunk 数远大于文档数。

**核心不变量：**

- boundaries 非递减、范围合法；
- chunks 拼接与原始 bytes 严格相等；
- 非空块能严格 UTF-8 decode；
- 内部 boundary 必须指向 delimiter 开始。


In [5]:
cases = [
    "",
    "没有任何分隔符 🌍",
    SPECIAL,
    SPECIAL + SPECIAL,
    "甲" + SPECIAL + SPECIAL + "乙",
    SPECIAL + "Hello!",
    "The end" + SPECIAL,
    "你🌍" + SPECIAL + "世界" + SPECIAL + "再见",
]
for text in cases:
    raw = text.encode("utf-8")
    for n in (1, 2, 4, 12):
        with io.BytesIO(raw) as f:
            bs = find_chunk_boundaries(f, n, DELIMITER, read_ahead=3)
        assert bs[0] == 0 and bs[-1] == len(raw)
        assert bs == sorted(set(bs))
        parts = [raw[a:b] for a, b in zip(bs[:-1], bs[1:])]
        assert b"".join(parts) == raw
        assert all(raw[b:].startswith(DELIMITER) for b in bs[1:-1])
        assert all(part.decode("utf-8").encode("utf-8") == part for part in parts)
print("8 组语料 × 4 种 chunk 数：边界测试通过")

for bad_n, bad_delim, bad_ahead in [(0, DELIMITER, 4), (2, b"", 4), (2, DELIMITER, 0)]:
    try:
        find_chunk_boundaries(
            io.BytesIO(b"abc"), bad_n, bad_delim, read_ahead=bad_ahead
        )
        raise AssertionError("应拒绝非法参数")
    except ValueError:
        pass

print("非法参数防御：通过")

8 组语料 × 4 种 chunk 数：边界测试通过
非法参数防御：通过


## 5. Math：什么时候可以使用 `Counter` aggregation？

令 `D(text)` 表示按**特殊标记**隔离后的 ordinary spans，`P(span)` 表示 regex pre-token 序列。

在切块边界也是合法文档边界的条件下，计数满足：

$$
C_{\text{corpus}}(p)
= \sum_{\text{ordinary span } s} \#\{p \in P(s)\}
= \sum_{\text{chunk } c} C_c(p)
$$

这里的 `p` 是普通 pre-token，**不包括 special token 本身**。这才是 `Counter` 的局部可加性；并不是说 regex 可以对任何随机切块独立处理。


In [6]:
delimiter_text = "<|endoftext|>"
corpus = "你好 world<|endoftext|> wonderful cat<|endoftext|>🌍 你好"
corpus_bytes = corpus.encode("utf-8")


def ordinary_pretoken_counts(text):
    # 学习用的计数观察：不生成 merges、vocab，也不实现官方 train_bpe。
    counts = Counter()
    for ordinary_span in text.split(delimiter_text):
        counts.update(pretokenize(ordinary_span))
    return counts


whole_counts = ordinary_pretoken_counts(corpus)

with io.BytesIO(corpus_bytes) as file:
    offsets = find_chunk_boundaries(file, 8, delimiter_text.encode(), read_ahead=3)
chunks = [corpus_bytes[a:b].decode("utf-8") for a, b in zip(offsets, offsets[1:])]
local_counts = [ordinary_pretoken_counts(chunk) for chunk in chunks]
merged_counts = sum(local_counts, Counter())

assert merged_counts == whole_counts
print("安全切块:", offsets)
print("局部统计:", local_counts)
print("一致性:", merged_counts == whole_counts)


安全切块: [0, 12, 39, 63]
局部统计: [Counter({'你好': 1, ' world': 1}), Counter({' wonderful': 1, ' cat': 1}), Counter({'🌍': 1, ' 你好': 1})]
一致性: True


## 6. 从文件块得到局部 Counter

现在才引入 `count_chunk_bytes()`：在**已知 chunk 边界正确**的前提下，先 decode，再隔离 `<|endoftext|>`，最后为普通文本统计 `Counter`。

`Counter.update(...)` 是加法而不是覆盖：同一个 pre-token 在不同块中出现，频率会自然累加。


In [7]:
def count_chunk_bytes(chunk: bytes) -> PretokenCounts:
    return count_with_special_isolation(chunk.decode("utf-8"))


def count_chunks(raw: bytes, desired_num_chunks: int, read_ahead: int = 7):
    with io.BytesIO(raw) as f:
        boundaries = find_chunk_boundaries(
            f, desired_num_chunks, DELIMITER, read_ahead=read_ahead
        )
    parts = [raw[a:b] for a, b in zip(boundaries[:-1], boundaries[1:])]
    local_counters = [count_chunk_bytes(part) for part in parts]
    global_counter = Counter()
    for c in local_counters:
        global_counter.update(c)
    return boundaries, local_counters, global_counter


text = "hello 世界!" + SPECIAL + "hello 世界!" + SPECIAL + "hello 🌍"
bs, locals_, merged = count_chunks(text.encode("utf-8"), 5)
whole_counts = count_with_special_isolation(text)

assert merged == whole_counts

print("local Counter 大小:", [len(c) for c in locals_])
print("global == whole:", merged == whole_counts)

local Counter 大小: [3, 3, 2]
global == whole: True


## 7. 差分测试：全文件统计必须等于分块统计

05 和 06 保留 Reference Trainer 的思想，在这里同样适用。

先对完整字符串调用 `count_with_special_isolation(text)` 作为 oracle，再把相同输入写成 bytes 并随机改变 chunk 数与 `read_ahead`。关键是**比较整个 Counter**，而不是仅比较 pre-token 总数。


In [8]:
rng = random.Random(336)
vocabulary = ["cat", " dog", "世界", " 🌍", "\n", "a", "a a", "?", "你", "123", ""]

for trial in range(90):
    documents = [
        "".join(rng.choice(vocabulary) for _ in range(rng.randrange(1, 10)))
        for _ in range(rng.randrange(1, 9))
    ]
    text = SPECIAL.join(documents)
    oracle = count_with_special_isolation(text)
    _, _, actual = count_chunks(
        text.encode("utf-8"),
        desired_num_chunks=rng.randrange(1, 18),
        read_ahead=rng.randrange(1, 16),
    )

    assert actual == oracle, f"failed trial {trial}"

print("90 次随机分块与 reference Counter 差分测试：通过")

90 次随机分块与 reference Counter 差分测试：通过


## 8. 把内存实验迁移成真实文件输入

小文件演示可以一次性 `read()`，但大语料不应把全部 bytes 放在内存里。下面每次只读取一个 `[start, end)` 文件区间，再聚合 Counter。

这样形成了 `input_path → safe boundaries → chunk Counter → global Counter` 的闭环。实际工程若采用多进程，worker 只需获得路径和 offsets，然后返回局部 Counter。

In [ ]:
def count_file_by_chunks(path: str | Path, desired_num_chunks: int) -> PretokenCounts:
    path = Path(path)
    with path.open("rb") as f:
        boundaries = find_chunk_boundaries(f, desired_num_chunks, DELIMITER)
    total = Counter()
    with path.open("rb") as f:
        for start, end in zip(boundaries[:-1], boundaries[1:]):
            f.seek(start)
            local = count_chunk_bytes(f.read(end - start))
            total.update(local)
    return total


with tempfile.TemporaryDirectory() as d:
    path = Path(d) / "toy_corpus.txt"
    content = (("the cat 🌍" + SPECIAL + "a dog" + SPECIAL) * 40) + "尾声"
    path.write_bytes(content.encode("utf-8"))
    actual = count_file_by_chunks(path, desired_num_chunks=6)
    expected = count_with_special_isolation(content)

    assert actual == expected

    print("file bytes:", path.stat().st_size)
    print("unique pre-tokens:", len(actual))
    print("file Counter == in-memory Counter:", actual == expected)

file bytes: 1726
unique pre-tokens: 6
file Counter == in-memory Counter: True


## 9. 为什么 pre-tokenization 可并行，BPE merge 不能每块各训一次？

Counter 满足加法结合律：

$$
(C_1+C_2)+C_3 = C_1+(C_2+C_3).
$$

因此局部统计能够独立执行。下面用 ThreadPool 做**顺序无关性**演示；它只模拟并发调度，不代表多线程一定快。

但 BPE 每轮 winner 是由**全局**频率决定的，`argmax` 不满足“分别选择再相加”的性质。正确方式始终是先汇总全局 Counter，再做统一 merge。

In [10]:
text = SPECIAL.join(["a b a", "a b", "hello 世界", "a b a", " 🌍 !"])
raw = text.encode("utf-8")
with io.BytesIO(raw) as f:
    bds = find_chunk_boundaries(f, 5, DELIMITER, read_ahead=4)
pieces = [raw[a:b] for a, b in zip(bds[:-1], bds[1:])]

with ThreadPoolExecutor(max_workers=3) as pool:
    parallel_counters = list(pool.map(count_chunk_bytes, pieces))
combined = Counter()
for one in parallel_counters:
    combined.update(one)

assert combined == count_with_special_isolation(text)
print("并发顺序不影响全局 Counter:", combined == count_with_special_isolation(text))

并发顺序不影响全局 Counter: True


## 10. 将全局 Counter 接入教学版 BPE merge loop

为了明确衔接 05，这里写一个**最小可观察版本**。每轮统计所有局部 sequence 内的相邻 pair（用 frequency 加权），选 winner 后对所有 sequence 执行 merge。

这里只做少量轮次，以显示：全文件与分块聚合得到的 `ordered merges` 必须相同。

In [11]:
def count_pairs_weighted(counts: PretokenCounts):
    pair_counts = Counter()
    for tokens, frequency in counts.items():
        pair_counts.update(
            {
                pair: frequency * occurrences
                for pair, occurrences in Counter(zip(tokens, tokens[1:])).items()
            }
        )
    return pair_counts


def merge_sequence(tokens: TokenSequence, pair):
    result = []
    i = 0
    while i < len(tokens):
        if i + 1 < len(tokens) and (tokens[i], tokens[i + 1]) == pair:
            result.append(tokens[i] + tokens[i + 1])
            i += 2
        else:
            result.append(tokens[i])
            i += 1
    return tuple(result)


def train_small_bpe(counts: PretokenCounts, steps=4):
    state = Counter(counts)
    merges = []
    for _ in range(steps):
        pairs = count_pairs_weighted(state)
        if not pairs:
            break
        # tie-breaking 沿用 05：先频率，频率相同时按 pair 的 bytes 字典序。
        winner = max(pairs, key=lambda pair: (pairs[pair], pair))
        merges.append(winner)
        new_state = Counter()
        for sequence, freq in state.items():
            new_state[merge_sequence(sequence, winner)] += freq
        state = new_state
    return merges, state


text = SPECIAL.join([" hello hello", " hello 🌍", " hi hello", " hello"])
_, _, split_counts = count_chunks(text.encode("utf-8"), 7, read_ahead=2)
whole_counts = count_with_special_isolation(text)
assert split_counts == whole_counts
m1, s1 = train_small_bpe(whole_counts, steps=7)
m2, s2 = train_small_bpe(split_counts, steps=7)
assert m1 == m2 and s1 == s2
print("ordered merges:", m1)
print("merge sequence 和最终 state 逐项相同：通过")

ordered merges: [(b' ', b'h'), (b'l', b'o'), (b'l', b'lo'), (b'e', b'llo'), (b' h', b'ello'), (b'\xf0', b'\x9f'), (b'\xf0\x9f', b'\x8c')]
merge sequence 和最终 state 逐项相同：通过


## 11. 性能瓶颈与边界

到这里，我们解决的是 correctness。多进程可减少某些规模下的 pre-tokenization 墙钟时间，却不是免费优化：

- 每个 worker 会额外消耗内存，并需要传递/合并 Counter；
- 文件 I/O、进程启动、IPC 都有开销；
- 切分过细可能使性能更差，切分过粗又限制并行度；
- BPE merge 的全局依赖仍需要 06 的增量维护策略。

不能把“多进程能运行”当作“多进程结果正确”；**Reference 差分测试始终是先决条件**。
